# PaySim Fraud Baseline

Train a time-aware, class-balanced logistic regression baseline. Identifier columns are excluded, and `isFlaggedFraud` remains a documented rule signal for later ablation testing.

In [2]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'PS_20174392719_1491204439457_log.csv'
ARTIFACT_DIR = ROOT / 'backend' / 'ml' / 'artifacts' / 'paysim_baseline'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

feature_columns = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
]
frame = pd.read_csv(DATA_PATH, usecols=feature_columns + ['isFraud'])
frame['isFraud'] = frame['isFraud'].astype('int8')
train_frame = frame[frame['step'] <= 594].copy()
test_frame = frame[frame['step'] > 594].copy()

# Bound the training cost while retaining all positives in each chronological partition.
train_positive = train_frame[train_frame['isFraud'] == 1]
train_negative_pool = train_frame[train_frame['isFraud'] == 0]
train_negative = train_negative_pool.sample(n=min(750_000, len(train_negative_pool)), random_state=42)
test_positive = test_frame[test_frame['isFraud'] == 1]
test_negative_pool = test_frame[test_frame['isFraud'] == 0]
test_negative = test_negative_pool.sample(n=min(250_000, len(test_negative_pool)), random_state=42)
train_frame = pd.concat([train_positive, train_negative]).sample(frac=1, random_state=42)
test_frame = pd.concat([test_positive, test_negative]).sample(frac=1, random_state=42)

X_train = train_frame[feature_columns]
y_train = train_frame['isFraud']
X_test = test_frame[feature_columns]
y_test = test_frame['isFraud']
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1
    )),
])
pipeline.fit(X_train, y_train)
probabilities = pipeline.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype('int8')

metrics = {
    'model_name': 'paysim_logistic_regression',
    'model_version': '2026-07-23.baseline.1',
    'feature_columns': feature_columns,
    'time_split': {'train_max_step': 594, 'test_min_step': 595},
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'positive_train_rows': int(y_train.sum()),
    'positive_test_rows': int(y_test.sum()),
    'precision': float(precision_score(y_test, predictions, zero_division=0)),
    'recall': float(recall_score(y_test, predictions, zero_division=0)),
    'pr_auc': float(average_precision_score(y_test, probabilities)),
    'roc_auc': float(roc_auc_score(y_test, probabilities)),
    'confusion_matrix': confusion_matrix(y_test, predictions).tolist(),
}

joblib.dump(pipeline, ARTIFACT_DIR / 'model.joblib')
(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
metrics

{'model_name': 'paysim_logistic_regression',
 'model_version': '2026-07-23.baseline.1',
 'feature_columns': ['step',
  'amount',
  'oldbalanceOrg',
  'newbalanceOrig',
  'oldbalanceDest',
  'newbalanceDest',
  'isFlaggedFraud'],
 'time_split': {'train_max_step': 594, 'test_min_step': 595},
 'train_rows': 756559,
 'test_rows': 123580,
 'positive_train_rows': 6559,
 'positive_test_rows': 1654,
 'precision': 0.2117614954291172,
 'recall': 0.9383313180169287,
 'pr_auc': 0.7601329294715369,
 'roc_auc': 0.9880280228650197,
 'confusion_matrix': [[116149, 5777], [102, 1552]]}

In [3]:
credit_artifact = ROOT / 'backend' / 'ml' / 'artifacts' / 'creditcard_baseline'
paysim_artifact = ROOT / 'backend' / 'ml' / 'artifacts' / 'paysim_baseline'
credit_metrics = json.loads((credit_artifact / 'metrics.json').read_text(encoding='utf-8'))
paysim_metrics = json.loads((paysim_artifact / 'metrics.json').read_text(encoding='utf-8'))
credit_model = joblib.load(credit_artifact / 'model.joblib')
paysim_model = joblib.load(paysim_artifact / 'model.joblib')
assert credit_model.named_steps['classifier'].class_weight == 'balanced'
assert paysim_model.named_steps['classifier'].class_weight == 'balanced'
assert credit_metrics['feature_columns'][-1] == 'Amount'
assert paysim_metrics['time_split']['test_min_step'] > paysim_metrics['time_split']['train_max_step']
print({'creditcard_features': len(credit_metrics['feature_columns']), 'paysim_features': len(paysim_metrics['feature_columns']), 'artifacts': 'reloadable'})

{'creditcard_features': 30, 'paysim_features': 7, 'artifacts': 'reloadable'}
